# Model Promotion Notebook

This notebook handles the promotion of ML models from one environment to another (e.g., dev → test → prod).

## Process Overview:
1. **Environment Check**: Verify we're not running in development
2. **Parameter Validation**: Extract and validate input parameters
3. **Model Copy**: Copy the champion model to the destination registry
4. **Alias Update**: Set the new model as the active champion

## Prerequisites:
- Source model must exist and have a "champion" alias
- Destination model registry must be accessible
- Appropriate permissions for model registry operations

In [ ]:
# YAML STRUCTURE BREAKDOWN:
# - dev-commons:  CATALOG=pru,      ENV=dev,  DEST_MODEL_NAME=pru_prod.*
# - prod-commons: CATALOG=pru_prod, ENV=prod, MODEL_NAME=pru.*
# - uat-commons:  Similar pattern with environment-specific catalogs
# =============================================================================

# STEP 1: Import all required dependencies
import os
from typing import Optional
from databricks.feature_engineering import FeatureLookup, FeatureEngineeringClient
import mlflow
from mlflow.tracking.client import MlflowClient

print("📦 Dependencies imported successfully")

# STEP 2: Parameter extraction and validation
try:
    # Extract parameters from widgets (maps to look-up.yml variables)
    env = dbutils.widgets.get("ENV")  # variables.ENV.default
    src_model_name = dbutils.widgets.get("MODEL_NAME")  # variables.MODEL_NAME.default
    dst_model_name = dbutils.widgets.get(
        "DEST_MODEL_NAME"
    )  # variables.DEST_MODEL_NAME.default

    print("📋 EXTRACTED PARAMETERS:")
    print(f"   Environment: {env}")
    print(f"   Source Model: {src_model_name}")
    print(f"   Destination Model: {dst_model_name}")

except Exception as e:
    print(f"❌ Error extracting parameters: {str(e)}")
    dbutils.notebook.exit("Failed to extract required parameters")

# STEP 3: Environment validation (skip dev environments)
if env.lower() == "dev":
    print("⚠️  Model promotion skipped - development environment")
    print("   🔒 Promotion only allowed in test/uat/prod (per look-up.yml config)")
    dbutils.notebook.exit("Development environment - no promotion needed")


# STEP 4: Parameter validation function
def validate_parameters():
    """Validate parameters against look-up.yml requirements"""
    errors = []
    if not src_model_name or src_model_name.strip() == "":
        errors.append("MODEL_NAME is required")
    if not dst_model_name or dst_model_name.strip() == "":
        errors.append("DEST_MODEL_NAME is required")
    if not env or env.strip() == "":
        errors.append("ENV is required")

    if errors:
        error_msg = "❌ Parameter validation failed:\n" + "\n".join(
            [f"  • {error}" for error in errors]
        )
        print(error_msg)
        dbutils.notebook.exit("Invalid parameters")

    print("✅ Parameter validation successful")


validate_parameters()


# STEP 5: Initialize MLflow client
def initialize_mlflow_client():
    """Initialize MLflow client with Unity Catalog (databricks-uc registry)"""
    try:
        registry_uri = "databricks-uc"
        client = MlflowClient(registry_uri=registry_uri)
        mlflow.set_registry_uri(registry_uri)
        print(f"✅ MLflow client initialized with registry: {registry_uri}")
        return client
    except Exception as e:
        error_msg = f"Failed to initialize MLflow client: {str(e)}"
        print(f"❌ {error_msg}")
        dbutils.notebook.exit(error_msg)


mlflow_client = initialize_mlflow_client()


# STEP 6: Configuration class (maps to look-up.yml structure)
class ModelPromotionConfig:
    """Configuration based on look-up.yml environment patterns"""

    CHAMPION_ALIAS = "Champion"
    SOURCE_ALIAS = "champion"

    @staticmethod
    def get_destination_model_name(
        env: str, custom_dst_name: Optional[str] = None
    ) -> str:
        """Get destination model name (typically pru_prod.* for prod environments)"""
        return (
            custom_dst_name
            if custom_dst_name
            else f"pru_prod.pac_mlops.pac_mlops-model"
        )


config = ModelPromotionConfig()
final_dst_model_name = config.get_destination_model_name(env, dst_model_name)

print(f"📋 PROMOTION CONFIGURATION:")
print(f"   Source: {src_model_name}@{config.SOURCE_ALIAS}")
print(f"   Destination: {final_dst_model_name}")
print(f"   Target Alias: {config.CHAMPION_ALIAS}")


# STEP 7: Model promotion execution
def promote_model(
    client: MlflowClient, src_name: str, dst_name: str, src_alias: str = "champion"
) -> str:
    """Copy model from source to destination registry"""
    try:
        src_model_uri = f"models:/{src_name}@{src_alias}"
        print(f"🚀 Starting promotion: {src_model_uri} → {dst_name}")

        # Copy model version
        copied_model_version = client.copy_model_version(src_model_uri, dst_name)
        dest_version = copied_model_version.version
        print(f"✅ Model copied as version {dest_version}")
        return dest_version
    except Exception as e:
        error_msg = f"Model promotion failed: {str(e)}"
        print(f"❌ {error_msg}")
        raise Exception(error_msg)


# STEP 8: Set champion alias function
def set_champion_alias(
    client: MlflowClient, model_name: str, version: str, alias: str = "Champion"
) -> None:
    """Set the champion alias for promoted model"""
    try:
        print(f"👑 Setting {alias} alias for version {version}")
        client.set_registered_model_alias(name=model_name, alias=alias, version=version)
        print(f"✅ Model {model_name} v{version} set as {alias}")
    except Exception as e:
        error_msg = f"Failed to set champion alias: {str(e)}"
        print(f"❌ {error_msg}")
        raise Exception(error_msg)


# STEP 9: Execute promotion workflow
try:
    # Promote the model
    promoted_version = promote_model(
        client=mlflow_client,
        src_name=src_model_name,
        dst_name=final_dst_model_name,
        src_alias=config.SOURCE_ALIAS,
    )

    # Set champion alias
    set_champion_alias(
        client=mlflow_client,
        model_name=final_dst_model_name,
        version=promoted_version,
        alias=config.CHAMPION_ALIAS,
    )

    # STEP 10: Success summary
    print("=" * 60)
    print("🎉 MODEL PROMOTION COMPLETED SUCCESSFULLY!")
    print("=" * 60)
    print(f"📊 SUMMARY:")
    print(f"   Environment: {env.upper()}")
    print(f"   Source: {src_model_name}")
    print(f"   Destination: {final_dst_model_name}")
    print(f"   Promoted Version: {promoted_version}")
    print(f"   Champion Alias: {config.CHAMPION_ALIAS}")
    print("=" * 60)
    print(f"✅ Model {final_dst_model_name} v{promoted_version} is now active!")

    # Return success status
    dbutils.notebook.exit(
        {
            "status": "success",
            "promoted_model": final_dst_model_name,
            "promoted_version": promoted_version,
            "environment": env,
        }
    )

except Exception as e:
    print(f"❌ PROMOTION FAILED: {str(e)}")
    dbutils.notebook.exit(str(e))